# Compare independently fitted molecular assays

Supply `rna.h5ad` and `protein.h5ad` for the same spots. RNA contains raw counts (or a counts
layer); protein `X` contains your prepared nonnegative measurements. Store registered spatial
coordinates in the RNA object's `obsm["spatial"]`. This workflow fits each assay independently,
then matches their spatial components for display.

The study's RCC data came from [10x Genomics Xenium Protein FFPE Human Renal Carcinoma](https://www.10xgenomics.com/datasets/xenium-protein-ffpe-human-renal-carcinoma).
[Study illustration](https://dsong-lab.github.io/spDecon/_static/figures/rcc_structures.png).


## 1. Configure your input

Install the notebook dependencies, then edit `DATA_DIR`, the input filenames and `K` below.
Supply your own `.h5ad` files with unique spot and feature IDs. RNA fitting uses raw counts;
set `COUNTS_LAYER="counts"` when they are stored in a layer. Protein inputs should contain
your prepared nonnegative assay measurements. Each fitted spot needs positive total signal.

Running the fitting cells trains a new model. `DEVICE` accepts `"auto"`, `"cpu"` or `"cuda"`.
The study's reference-free and joint analyses used GPUs. Inspect the returned diagnostics
to determine whether your fit converged. Large inputs require enough memory for dense matrices.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from IPython.display import display
from tutorials.common import correlation, align, spatial, pies, legend, save_figure, save_table
from spDecon.reproducibility import set_seed

DATA_DIR = Path("/path/to/your/data")
COUNTS_LAYER = None
DEVICE = "auto"
VERBOSE = True
set_seed(1, deterministic=True)

K = 9  # Choose the component number for your data.
SPATIAL_WEIGHT = .05  # Choose for your analysis.


## 2. Align paired observations


In [ ]:
rna_adata = ad.read_h5ad(DATA_DIR / "rna.h5ad")
protein_adata = ad.read_h5ad(DATA_DIR / "protein.h5ad")
if (not rna_adata.obs_names.is_unique or not protein_adata.obs_names.is_unique
        or set(rna_adata.obs_names) != set(protein_adata.obs_names)):
    raise ValueError("Paired assays need identical, unique spot IDs")
protein_adata = protein_adata[rna_adata.obs_names].copy()
xy = np.asarray(rna_adata.obsm["spatial"])
if "spatial" in protein_adata.obsm:
    np.testing.assert_allclose(protein_adata.obsm["spatial"], xy)
sections = rna_adata.obs["section_id"].to_numpy() if "section_id" in rna_adata.obs else np.repeat("section_1", len(xy))
assays = {"RNA": rna_adata.to_df(layer=COUNTS_LAYER), "protein": protein_adata.to_df()}
print({name: values.shape for name, values in assays.items()})


## 3. Fit each assay separately

Each assay has its own exposure, starting factors and molecular profiles. The shared graph
represents registered physical locations. The configured spatial coefficient is applied to each fit.


In [ ]:
from spDecon.config import SolverConfig
from spDecon.consensus import build_consensus
from spDecon.spatial import spatial_graph, refine
i, j, w, _ = spatial_graph(xy, sections)
fits = {}
for name, frame in assays.items():
    y = np.array(frame, dtype=float, order="F")
    initial = build_consensus(y, K, random_state=1,
        solver_config=SolverConfig(device=DEVICE, random_state=1, verbose=VERBOSE))
    result = refine(y, initial.consensus_proportions, initial.consensus_profiles,
        i, j, w, SPATIAL_WEIGHT, 1, device=DEVICE, verbose=VERBOSE)
    fits[name] = result.proportions
    display(result.diagnostics)


## 4. Match and display spatial components

Hungarian assignment maximizes spatial weight correlation across the supplied spots. These
display correlations describe cross-assay correspondence; neither assay supplies cell-type truth.


In [ ]:
from scipy.optimize import linear_sum_assignment
matrix = correlation(fits["RNA"], fits["protein"])
rows, cols = linear_sum_assignment(-matrix)
order = cols[np.argsort(rows)]
pairs = pd.DataFrame({"RNA_component": np.arange(1, K+1),
    "protein_component": order+1, "display_PCC": matrix[np.arange(K), order]})
display(save_table(pairs, "cross_modality_matching"))
fig, axes = plt.subplots(2, K, figsize=(2.3*K, 5), squeeze=False, layout="constrained")
for k, j in enumerate(order):
    spatial(axes[0,k], xy, fits["RNA"][:,k], f"RNA C{k+1}", size=10)
    art = spatial(axes[1,k], xy, fits["protein"][:,j], f"Protein C{j+1}", size=10)
fig.colorbar(art, ax=axes, label="Molecular contribution", shrink=.6)
save_figure(fig, "cross_modality_components")
out = ROOT / "outputs/cross_modality"
out.mkdir(parents=True, exist_ok=True)
for name, weights in fits.items():
    pd.DataFrame(weights, index=rna_adata.obs_names,
        columns=[f"C{k+1}" for k in range(K)]).to_csv(out / f"{name}_weights.csv")
